# Detecting Topic Merges and Splits in Dynamic Political Conversations

Cláudia Oliveira 

Supervisor - Prof. Dr. Álvaro Figueira

Faculty of Science, University of Porto

In [ ]:
import pandas as pd
import string
import re
import spacy
from sklearn.feature_extraction.text import CountVectorizer
import os
os.chdir("../..")

# -------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------
df = pd.read_csv("./datasets/un-general-debates.csv")
df = df.sort_values("year").reset_index(drop=True)

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
stops = nlp.Defaults.stop_words

# -------------------------------------------------------
# 2. GENERAL CLEANING + LEMMATIZATION + STOPWORDS
# -------------------------------------------------------
def clean_and_lemmatize(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation + "0123456789"))
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", text)

    doc = nlp(text)

    return [
        tok.lemma_
        for tok in doc
        if tok.lemma_.strip() != ""
        and len(tok.lemma_) > 1
        and tok.lemma_ not in stops
    ]

# -------------------------------------------------------
# 3. FIRST PASS — PROCESS THE FULL SPEECH
# -------------------------------------------------------
df["Tokens_full"] = df["text"].apply(clean_and_lemmatize)

docs_as_text = df["Tokens_full"].apply(lambda t: " ".join(t))

vectorizer = CountVectorizer()
doc_term_matrix = vectorizer.fit_transform(docs_as_text)
vocab = vectorizer.get_feature_names_out()

dfreq = (doc_term_matrix > 0).sum(axis=0).A1
doc_freq = pd.Series(dfreq, index=vocab)

N_docs = len(df)

# -------------------------------------------------------
# 4. FREQUENCY FILTERS
# -------------------------------------------------------
min_docs = 10
max_docs = 0.7 * N_docs   

allowed_vocab = set(doc_freq[(doc_freq >= min_docs) & (doc_freq <= max_docs)].index)

print("Final vocabulary size:", len(allowed_vocab))

# -------------------------------------------------------
# 5. SECOND PASS — PROCESS PARAGRAPH BY PARAGRAPH
# -------------------------------------------------------

def preprocess_paragraph(text):
    # use the same cleaning + stopwords removal
    tokens = clean_and_lemmatize(text)
    # filter by allowed vocabulary
    tokens = [t for t in tokens if t in allowed_vocab]
    return tokens

all_years = []
all_tokens = []
all_text = []

for _, row in df.iterrows():
    year = row["year"]
    raw_text = row["text"]

    # split text into paragraphs only now
    paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

    for p in paragraphs:
        tokens = preprocess_paragraph(p)
        if tokens:
            all_years.append(year)
            all_tokens.append(tokens)
            all_text.append(" ".join(tokens))

# -------------------------------------------------------
# 6. FINAL DATAFRAME — SAME FORMAT AS BEFORE
# -------------------------------------------------------
df_processed = pd.DataFrame({
    "Year": all_years,
    "Tokens": all_tokens
})

print("Final number of paragraphs:", len(df_processed))
print(df_processed.head())

Final vocabulary size: 25464
Final number of paragraphs: 26362
   Year                                             Tokens  \
0  1970  [fortunate, anniversary, distinguished, citize...   
1  1970  [pleasure, extend, sincere, congratulation, li...   
2  1970  [behalf, excellency, mzee, jomo, kenyatta, rep...   
3  1970  [pleasure, speak, preside, meeting, obliged, t...   
4  1970  [delegation, malaysia, philippine, delegation,...   

                                                Text  
0  fortunate anniversary distinguished citizen no...  
1  pleasure extend sincere congratulation liberia...  
2  behalf excellency mzee jomo kenyatta republic ...  
3  pleasure speak preside meeting obliged transmi...  
4  delegation malaysia philippine delegation shar...  


In [5]:
df_processed.to_csv('datasets/UN_General_Debates_Tokens.csv', index=False)

PARAGRAPHS PER YEAR

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("datasets/UN_General_Debates_Tokens.csv")
counts = df.groupby("Year").size().reset_index(name="Count")

plt.figure(figsize=(10,5))
plt.plot(counts["Year"], counts["Count"])
plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Number of Paragraphs per Year")
plt.tight_layout()
plt.xticks(counts["Year"], rotation=90)
plt.show()